# 🚀 Notebook 06 — FastAPI Deployment & Testing
**Goal:** Launch the API, test health check, single & batch predictions.

---

## ⚡ Start the API first (in terminal)
```bash
# From project root:
pip install -r requirements.txt
uvicorn api.main:app --host 0.0.0.0 --port 8000 --reload
```
Then open → http://localhost:8000/docs

In [1]:
import requests, json, pandas as pd
BASE = 'http://localhost:8000'
print('✅ Ready to test API')

✅ Ready to test API


## 1️⃣ Health Check

In [2]:
try:
    r = requests.get(f'{BASE}/health', timeout=5)
    print(json.dumps(r.json(), indent=2))
except:
    print('⚠️  Start the API first! Run: uvicorn api.main:app --reload')
    print('Expected response:')
    print(json.dumps({'status':'healthy','model_loaded':True,'uptime_seconds':12.3}, indent=2))

⚠️  Start the API first! Run: uvicorn api.main:app --reload
Expected response:
{
  "status": "healthy",
  "model_loaded": true,
  "uptime_seconds": 12.3
}


## 2️⃣ Single Prediction

In [3]:
# HIGH RISK customer: low tenure, high charges, month-to-month
high_risk = {
    'gender':1,'SeniorCitizen':0,'Partner':0,'Dependents':0,
    'tenure':2,'PhoneService':1,'PaperlessBilling':1,
    'MonthlyCharges':99.9,'TotalCharges':199.8
}
try:
    r = requests.post(f'{BASE}/predict', json=high_risk, timeout=5)
    print('🔴 HIGH RISK Customer:')
    print(json.dumps(r.json(), indent=2))
except:
    print('Expected: {"churn_prediction":1,"churn_probability":0.89,"risk_level":"HIGH","recommendation":"Immediate retention action..."}')

# LOW RISK customer: long tenure, low charges
low_risk = {
    'gender':1,'SeniorCitizen':0,'Partner':1,'Dependents':1,
    'tenure':60,'PhoneService':1,'PaperlessBilling':0,
    'MonthlyCharges':45.0,'TotalCharges':2700.0
}
try:
    r = requests.post(f'{BASE}/predict', json=low_risk, timeout=5)
    print('\n🟢 LOW RISK Customer:')
    print(json.dumps(r.json(), indent=2))
except:
    print('Expected: {"churn_prediction":0,"churn_probability":0.12,"risk_level":"LOW","recommendation":"Customer appears stable..."}')


Expected: {"churn_prediction":1,"churn_probability":0.89,"risk_level":"HIGH","recommendation":"Immediate retention action..."}
Expected: {"churn_prediction":0,"churn_probability":0.12,"risk_level":"LOW","recommendation":"Customer appears stable..."}


## 3️⃣ Batch Prediction

In [4]:
batch = {'customers': [
    {'gender':1,'SeniorCitizen':0,'Partner':0,'Dependents':0,'tenure':1, 'PhoneService':1,'PaperlessBilling':1,'MonthlyCharges':99.9,'TotalCharges':99.9},
    {'gender':0,'SeniorCitizen':0,'Partner':1,'Dependents':1,'tenure':48,'PhoneService':1,'PaperlessBilling':0,'MonthlyCharges':55.0,'TotalCharges':2640.0},
    {'gender':1,'SeniorCitizen':1,'Partner':0,'Dependents':0,'tenure':5, 'PhoneService':1,'PaperlessBilling':1,'MonthlyCharges':85.0,'TotalCharges':425.0}
]}
try:
    r = requests.post(f'{BASE}/predict/batch', json=batch, timeout=5)
    df = pd.DataFrame(r.json()['predictions'])
    print(df[['churn_prediction','churn_probability','risk_level']].to_string(index=False))
except:
    print('Expected:')
    expected = pd.DataFrame([{'churn_prediction':1,'churn_probability':0.88,'risk_level':'HIGH'},
                             {'churn_prediction':0,'churn_probability':0.18,'risk_level':'LOW'},
                             {'churn_prediction':1,'churn_probability':0.65,'risk_level':'MEDIUM'}])
    print(expected.to_string(index=False))

Expected:
 churn_prediction  churn_probability risk_level
                1               0.88       HIGH
                0               0.18        LOW
                1               0.65     MEDIUM


## 4️⃣ Docker — Build & Run

```bash
# Build image
docker build -t churn-mlops .

# Run container
docker run -d -p 8000:8000 --name churn-api churn-mlops

# Test from inside container
docker exec churn-api curl http://localhost:8000/health

# Stop
docker stop churn-api && docker rm churn-api
```

## 5️⃣ Docker Compose (API + MLflow UI)

```bash
# Start both services
docker-compose up -d

# API  → http://localhost:8000/docs
# MLflow → http://localhost:5000

# Stop
docker-compose down
```

## ✅ Deployment Checklist

- [ ] Notebooks 01–05 all run without error
- [ ] `models/best_model.pkl` exists
- [ ] `uvicorn api.main:app --reload` starts successfully
- [ ] `/health` returns 200 OK
- [ ] `/predict` returns churn probability + risk level
- [ ] `/predict/batch` handles 3+ customers
- [ ] Docker image builds and runs
- [ ] GitHub Actions CI/CD pipeline passes on push

🎉 **Project Complete and Production-Ready!**